# Textvorverarbeitung

In diesem Notebook werden die Review-Texte bereinigt und für die anschließende Vektorisierung und das Topic Modelling vorbereitet.

In [1]:
import pandas as pd

file_path = "../data/Grand_Theft_Auto_V.jsonlines"

df = pd.read_json(file_path, lines=True)

df_clean = df.drop_duplicates(subset=["review"]).copy()

df_clean = df_clean[["rating", "review"]].copy()

df_clean.shape

(11190, 2)

## Bereinigung eines Beispieltexts

In [ ]:
example_review = df_clean["review"].iloc[0]

print(example_review)

In [ ]:
example_lower = example_review.lower()

print(example_lower)

In [ ]:
import re

example_cleaned = re.sub(r"[^a-z\s]", " ", example_lower)
example_cleaned = re.sub(r"\s+", " ", example_cleaned).strip()

print(example_cleaned)

In [5]:
example_tokens = example_cleaned.split()

print(example_tokens)

['well', 'hackers', 'got', 'to', 'the', 'point', 'where', 'they', 'use', 'my', 'name', 'to', 'kill', 'all', 'others', 'on', 'the', 'server', 'so', 'people', 'think', 'its', 'me', 'who', 'is', 'hacking', 'besides', 'that', 'the', 'hacker', 'succeeded', 'in', 'putting', 'me', 'in', 'a', 'death', 'loop', 'where', 'i', 'endlessly', 'die', 'on', 'top', 'of', 'that', 'he', 'succesfully', 'crashed', 'my', 'game', 'probably', 'with', 'an', 'invalid', 'texture', 'input', 'eventhough', 'i', 'have', 'lots', 'of', 'fun', 'playing', 'this', 'is', 'an', 'issue', 'that', 'needs', 'to', 'be', 'solved']


In [ ]:
import nltk

nltk.download("stopwords")

In [7]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

example_tokens_filtered = [
    token
    for token in example_tokens
    if token not in stop_words
]

print(example_tokens_filtered)

['well', 'hackers', 'got', 'point', 'use', 'name', 'kill', 'others', 'server', 'people', 'think', 'hacking', 'besides', 'hacker', 'succeeded', 'putting', 'death', 'loop', 'endlessly', 'die', 'top', 'succesfully', 'crashed', 'game', 'probably', 'invalid', 'texture', 'input', 'eventhough', 'lots', 'fun', 'playing', 'issue', 'needs', 'solved']


In [ ]:
nltk.download("wordnet")
nltk.download("omw-1.4")

In [9]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

example_tokens_lemmatized = [
    lemmatizer.lemmatize(token)
    for token in example_tokens_filtered
]

print(example_tokens_lemmatized)

['well', 'hacker', 'got', 'point', 'use', 'name', 'kill', 'others', 'server', 'people', 'think', 'hacking', 'besides', 'hacker', 'succeeded', 'putting', 'death', 'loop', 'endlessly', 'die', 'top', 'succesfully', 'crashed', 'game', 'probably', 'invalid', 'texture', 'input', 'eventhough', 'lot', 'fun', 'playing', 'issue', 'need', 'solved']


In [10]:
def preprocess_text(text):
    # Text in Kleinbuchstaben umwandeln
    text = text.lower()

    # Zahlen, Satzzeichen und Sonderzeichen entfernen
    text = re.sub(r"[^a-z\s]", " ", text)

    # Mehrere Leerzeichen zusammenfassen
    text = re.sub(r"\s+", " ", text).strip()

    # Text in einzelne Wörter zerlegen
    tokens = text.split()

    # Stopwörter entfernen
    tokens = [
        token
        for token in tokens
        if token not in stop_words
    ]

    # Wörter lemmatisieren
    tokens = [
        lemmatizer.lemmatize(token)
        for token in tokens
    ]

    return tokens

In [11]:
preprocess_text(example_review)

['well',
 'hacker',
 'got',
 'point',
 'use',
 'name',
 'kill',
 'others',
 'server',
 'people',
 'think',
 'hacking',
 'besides',
 'hacker',
 'succeeded',
 'putting',
 'death',
 'loop',
 'endlessly',
 'die',
 'top',
 'succesfully',
 'crashed',
 'game',
 'probably',
 'invalid',
 'texture',
 'input',
 'eventhough',
 'lot',
 'fun',
 'playing',
 'issue',
 'need',
 'solved']

## Anwendung auf den gesamten Datensatz

Die am Beispiel dargestellten Verarbeitungsschritte werden nun auf alle eindeutigen Review-Texte angewendet. Für jeden Review werden die bereinigten Tokens sowie deren Anzahl gespeichert.

In [12]:
df_clean["tokens"] = df_clean["review"].apply(preprocess_text)

In [ ]:
df_clean[["review", "tokens"]].head()

## Prüfung der verwertbaren Textlänge

Nach der Vorverarbeitung wird geprüft, wie viele verwertbare Tokens je Review verbleiben. Leere Texte können nicht vektorisiert werden. Reviews mit nur einem Token werden ebenfalls ausgeschlossen, da sie für die Identifikation zusammenhängender Themen nur eine sehr geringe Informationsgrundlage bieten. Reviews ab zwei Tokens bleiben erhalten, um den Datensatz nicht unnötig stark zu reduzieren.

In [14]:
df_clean["token_count"] = df_clean["tokens"].str.len()

df_clean["token_count"].describe()

count    11190.000000
mean        32.148257
std         58.423030
min          0.000000
25%          6.000000
50%         13.000000
75%         33.000000
max       1067.000000
Name: token_count, dtype: float64

In [15]:
(df_clean["token_count"] == 0).sum()

np.int64(7)

In [ ]:
df_clean.loc[
    df_clean["token_count"] == 0,
    ["rating", "review", "tokens"]
]

In [17]:
df_processed = df_clean[
    df_clean["token_count"] > 0
].copy()

df_processed = df_processed.reset_index(drop=True)

df_processed.shape

(11183, 4)

In [18]:
processed_length_counts = {
    "1 Token": (df_processed["token_count"] == 1).sum(),
    "höchstens 3 Tokens": (df_processed["token_count"] <= 3).sum(),
    "höchstens 5 Tokens": (df_processed["token_count"] <= 5).sum(),
    "höchstens 10 Tokens": (df_processed["token_count"] <= 10).sum()
}

processed_length_counts

{'1 Token': np.int64(195),
 'höchstens 3 Tokens': np.int64(1241),
 'höchstens 5 Tokens': np.int64(2521),
 'höchstens 10 Tokens': np.int64(4813)}

In [19]:
df_processed["token_count"].value_counts().sort_index().head(11)

token_count
1     195
2     484
3     562
4     671
5     609
6     601
7     489
8     419
9     405
10    378
11    362
Name: count, dtype: int64

In [ ]:
df_processed.loc[
    df_processed["token_count"] <= 3,
    ["rating", "review", "tokens", "token_count"]
].sample(15, random_state=42).sort_values("token_count")

In [21]:
df_model = df_processed[
    df_processed["token_count"] >= 2
].copy()

df_model = df_model.reset_index(drop=True)

df_model.shape

(10988, 4)

In [22]:
df_model["clean_text"] = df_model["tokens"].apply(" ".join)

In [ ]:
df_model[["review", "tokens", "clean_text"]].head()

## Häufigste Wörter nach der Textvorverarbeitung

In [24]:
from collections import Counter

all_tokens = [
    token
    for tokens in df_model["tokens"]
    for token in tokens
]

word_frequencies = Counter(all_tokens)

word_frequencies.most_common(30)

[('game', 16148),
 ('gta', 4412),
 ('online', 3903),
 ('rockstar', 3562),
 ('get', 3062),
 ('play', 2974),
 ('pc', 2921),
 ('like', 2557),
 ('time', 2468),
 ('great', 2239),
 ('good', 2173),
 ('even', 2029),
 ('one', 2028),
 ('mod', 2019),
 ('player', 1982),
 ('fun', 1976),
 ('would', 1927),
 ('people', 1809),
 ('really', 1618),
 ('buy', 1572),
 ('v', 1569),
 ('story', 1524),
 ('loading', 1467),
 ('money', 1451),
 ('best', 1451),
 ('well', 1404),
 ('much', 1388),
 ('run', 1380),
 ('still', 1371),
 ('graphic', 1371)]

Die Ergebnisse zeigen, dass sowohl allgemeine spielbezogene Begriffe wie `game` und `play` als auch spezifische Begriffe wie `rockstar`, `online`, `mod`, `story` und `loading` häufig auftreten. Damit sind bereits erste mögliche Themenbereiche erkennbar.

In [25]:
document_frequencies = Counter(
    token
    for tokens in df_model["tokens"]
    for token in set(tokens)
)

document_frequencies.most_common(30)

[('game', 6777),
 ('gta', 2474),
 ('rockstar', 2447),
 ('online', 2243),
 ('play', 2071),
 ('get', 2002),
 ('pc', 1931),
 ('like', 1673),
 ('great', 1645),
 ('time', 1641),
 ('good', 1635),
 ('would', 1522),
 ('even', 1481),
 ('fun', 1462),
 ('one', 1417),
 ('buy', 1280),
 ('player', 1223),
 ('best', 1218),
 ('people', 1205),
 ('mod', 1168),
 ('v', 1128),
 ('graphic', 1121),
 ('really', 1120),
 ('much', 1065),
 ('well', 1062),
 ('still', 1062),
 ('money', 1060),
 ('run', 1052),
 ('story', 1029),
 ('worth', 1006)]

## Ergebnis der Textvorverarbeitung

Nach der Entfernung identischer Texte, leerer Ergebnisse und Ein-Token-Reviews umfasst der aufbereitete Datensatz 10.988 Reviews. Die bereinigten Texte werden für die anschließende Vektorisierung als JSONLines-Datei gespeichert.

In [26]:
from pathlib import Path

output_path = Path("../data/processed/processed_reviews.jsonlines")

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

df_model.to_json(
    output_path,
    orient="records",
    lines=True,
    force_ascii=False
)

print(f"Datei gespeichert unter: {output_path}")

Datei gespeichert unter: ..\data\processed\processed_reviews.jsonlines
